This model uses a Convolutional Neural Network design in order to classify images, using various convolutions that are developed and refined overtime across several training sessions in order to properly classify images into various categories.

The programming will be done in two ways. First, pytorch will be used in order to build a simple CNN. Then, it'll be done from scratch using numpy as the mathematics library.

Training data will be stored in a folder named "Datasets" and in two folders - "Cat" and "Dog". The dataset was obtained from here: https://www.kaggle.com/datasets/bhavikjikadara/dog-and-cat-classification-dataset?resource=download.

In [39]:
# Initial Imports.
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import numpy as np
import os

In [40]:
# Setting torch to use GPU acceleration if possible.
device = torch.device("cpu")

if torch.cuda.is_available():
    device = torch.device("cuda")

torch.set_default_device(device)
print(f"Using device: {torch.get_default_device()}")

Using device: cpu


**Pytorch Model**

In [41]:
# Preparing dataset for Pytorch
class PytorchDataset(Dataset):
    def __init__(self, root = "Datasets"):
        self.transform = transforms.Compose({
            transforms.Resize((32, 32)),
            transforms.ToTensor()
        })

        self.images = []
        self.labels = []

        self.classes = {
            "Cat": 0,
            "Dog": 1
        }

        # Load dataset
        for className, label in self.classes.items():
            classPath = os.path.join(root, className)

            for imageName in os.listdir(classPath):
                imagePath = os.path.join(classPath, imageName)
                self.images.append(imagePath)
                self.labels.append(label)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        imagePath = self.images[idx]

        # Load and prepare image.
        image = Image.open(imagePath).convert("RGB")
        image = self.transform(image)

        label = self.labels[idx]

        return image, label

In [42]:
# Generate the dataset. This dataset can then be used in as many models as we want.
dataset = PytorchDataset()

In [43]:
class PytorchModel(nn.Module):
    def __init__(self):
        super(PytorchModel, self).__init__()
        self.dataset = dataset
        
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.l1 = nn.Linear(16 * 5 * 5, 120)
        self.l2 = nn.Linear(120, 84)
        self.l3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.l1(x))
        x = F.relu(self.l2(x))
        return self.l3(x)

In [49]:
# Prepare Training.
model = PytorchModel()

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr = 0.001, momentum = 0.9)

trainLoader = DataLoader(dataset, batch_size = 32, shuffle = True)

In [50]:
# Train the model.
epochs = 50

for epoch in range(epochs):
    runningLoss = 0.0

    for i, data in enumerate(trainLoader):
        inputs, labels = data
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        runningLoss += loss.item()
        
    print("Epoch #" + str(epoch + 1) + ": " + str(runningLoss / (i + 1)))

Epoch #1: 0.8569628706063761
Epoch #2: 0.6972201862146178
Epoch #3: 0.6916871057142078
Epoch #4: 0.6879117887495728
Epoch #5: 0.6834235925351262
Epoch #6: 0.676551914428506
Epoch #7: 0.6719291611858036
Epoch #8: 0.6694031382155845
Epoch #9: 0.6654498059578868
Epoch #10: 0.6616954214280218
Epoch #11: 0.6584751423057693
Epoch #12: 0.6528989552232005
Epoch #13: 0.6486266833132185
Epoch #14: 0.6452107779738848
Epoch #15: 0.6386637788294526
Epoch #16: 0.6351612894736287
Epoch #17: 0.6291092509961189
Epoch #18: 0.6253044179181004
Epoch #19: 0.6178557050731176
Epoch #20: 0.6121788419344846
Epoch #21: 0.6074909523243794
Epoch #22: 0.5956708235890055
Epoch #23: 0.587616338944801
Epoch #24: 0.579742373693782
Epoch #25: 0.5726036923696928
Epoch #26: 0.560346063140713
Epoch #27: 0.5535015456206963
Epoch #28: 0.5479954588596169
Epoch #29: 0.5394586218744898
Epoch #30: 0.5330769811825984
Epoch #31: 0.5287493561844692
Epoch #32: 0.5204749433966853
Epoch #33: 0.5128717956030765
Epoch #34: 0.5068889599

In [46]:
# Test the model.
testDataset = PytorchDataset(os.path.join("Datasets", "_TEST_"))
testLoader = DataLoader(testDataset, batch_size = 4, shuffle = False)

correct = 0
total = 0

with torch.no_grad():
    for data in testLoader:
        images, labels = data
        outputs = model(images)

        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Accuracy: " + str(100 * correct / total))

Accuracy: 59.8
